In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import itertools
try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


#### Constants

In [2]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

Bucket: assessment-swish-analytics
Task: 04_model


#### Import Data from S3

In [3]:
X_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_test.csv')

y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv').squeeze()
y_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_valid.csv').squeeze()
y_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_test.csv').squeeze()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


X_train: (538294, 90), y_train: (538294,)
X_valid: (120479, 90), y_valid: (120479,)
X_test:  (39545, 90), y_test:  (39545,)


#### Encode Target Labels

In [4]:
le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc = le.transform(y_test)

print(f'Classes: {le.classes_}')
print(f'Number of classes: {len(le.classes_)}')

Classes: ['CH' 'CU' 'FC' 'FF' 'FS' 'FT' 'SI' 'SL']
Number of classes: 8


#### Create DMatrices

In [5]:
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dvalid = xgb.DMatrix(X_valid, label=y_valid_enc)
dtest = xgb.DMatrix(X_test, label=y_test_enc)

#### Hyperparameter Tuning

Grid search over key hyperparameters using early stopping on the validation set. The best configuration (lowest validation `mlogloss`) is selected for the final model.

In [6]:
# fixed parameters
dict_fixed = {
    'objective': 'multi:softprob',
    'num_class': len(le.classes_),
    'eval_metric': 'mlogloss',
    'subsample': 0.8,
    'reg_lambda': 0.5,
    'gamma': 0,
    'seed': 42,
    'verbosity': 0,
}

# hyperparameter grid (small, focused search)
dict_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [5, 10],
}

# generate all combinations
keys = list(dict_grid.keys())
list_configs = list(itertools.product(*dict_grid.values()))
print(f'Total configurations to evaluate: {len(list_configs)}')

Total configurations to evaluate: 16


In [7]:
%%time

list_results = []
for i, values in enumerate(list_configs):
    dict_params_search = dict_fixed.copy()
    dict_params_search.update(dict(zip(keys, values)))
    
    model_tmp = xgb.train(
        dict_params_search,
        dtrain,
        num_boost_round=500,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=25,
        verbose_eval=False,
    )
    
    dict_result = dict(zip(keys, values))
    dict_result['best_iteration'] = model_tmp.best_iteration
    dict_result['best_mlogloss'] = model_tmp.best_score
    list_results.append(dict_result)
    
    print(f'  [{i+1}/{len(list_configs)}] lr={values[0]}, depth={values[1]}, '
          f'colsample={values[2]}, mcw={values[3]} -> '
          f'iter={model_tmp.best_iteration}, mlogloss={model_tmp.best_score:.4f}')

df_results = pd.DataFrame(list_results).sort_values('best_mlogloss')
print(f'\nTop 10 configurations:')
df_results.head(10)

  [1/16] lr=0.05, depth=4, colsample=0.7, mcw=5 -> iter=10, mlogloss=2.0058
  [2/16] lr=0.05, depth=4, colsample=0.7, mcw=10 -> iter=10, mlogloss=2.0058
  [3/16] lr=0.05, depth=4, colsample=0.9, mcw=5 -> iter=10, mlogloss=2.0078
  [4/16] lr=0.05, depth=4, colsample=0.9, mcw=10 -> iter=10, mlogloss=2.0078
  [5/16] lr=0.05, depth=6, colsample=0.7, mcw=5 -> iter=10, mlogloss=2.0063
  [6/16] lr=0.05, depth=6, colsample=0.7, mcw=10 -> iter=10, mlogloss=2.0063
  [7/16] lr=0.05, depth=6, colsample=0.9, mcw=5 -> iter=10, mlogloss=2.0083
  [8/16] lr=0.05, depth=6, colsample=0.9, mcw=10 -> iter=10, mlogloss=2.0083
  [9/16] lr=0.1, depth=4, colsample=0.7, mcw=5 -> iter=5, mlogloss=2.0062
  [10/16] lr=0.1, depth=4, colsample=0.7, mcw=10 -> iter=5, mlogloss=2.0062
  [11/16] lr=0.1, depth=4, colsample=0.9, mcw=5 -> iter=4, mlogloss=2.0088
  [12/16] lr=0.1, depth=4, colsample=0.9, mcw=10 -> iter=4, mlogloss=2.0088
  [13/16] lr=0.1, depth=6, colsample=0.7, mcw=5 -> iter=5, mlogloss=2.0070
  [14/16] lr

,learning_rate,max_depth,colsample_bytree,min_child_weight,best_iteration,best_mlogloss
1,0.05,4,0.7,10,10,2.005754
0,0.05,4,0.7,5,10,2.005758
9,0.10,4,0.7,10,5,2.006246
8,0.10,4,0.7,5,5,2.006247
5,0.05,6,0.7,10,10,2.006340
4,0.05,6,0.7,5,10,2.006341
12,0.10,6,0.7,5,5,2.006989
13,0.10,6,0.7,10,5,2.006999
3,0.05,4,0.9,10,10,2.007766
2,0.05,4,0.9,5,10,2.007768


#### Train Final Model with Best Hyperparameters

In [8]:
# extract best params from grid search
dict_best_row = df_results.iloc[0].to_dict()
flt_best_mlogloss = dict_best_row.pop('best_mlogloss')
int_best_iter_search = int(dict_best_row.pop('best_iteration'))

# cast int params (DataFrame stores them as float)
dict_best_row['max_depth'] = int(dict_best_row['max_depth'])
dict_best_row['min_child_weight'] = int(dict_best_row['min_child_weight'])

dict_params = dict_fixed.copy()
dict_params.update(dict_best_row)
dict_params['verbosity'] = 1

print(f'Best grid search mlogloss: {flt_best_mlogloss:.4f} at iteration {int_best_iter_search}')
print(f'\nBest parameters:')
for k, v in dict_params.items():
    print(f'  {k}: {v}')

# train final model
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=50,
)

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best validation mlogloss: {model.best_score:.4f}')

Best grid search mlogloss: 2.0058 at iteration 10

Best parameters:
  objective: multi:softprob
  num_class: 8
  eval_metric: mlogloss
  subsample: 0.8
  reg_lambda: 0.5
  gamma: 0
  seed: 42
  verbosity: 1
  learning_rate: 0.05
  max_depth: 4
  colsample_bytree: 0.7
  min_child_weight: 10
[0]	train-mlogloss:2.01742	valid-mlogloss:2.06376
[50]	train-mlogloss:1.21441	valid-mlogloss:2.23299
[60]	train-mlogloss:1.18058	valid-mlogloss:2.30975

Best iteration: 10
Best validation mlogloss: 2.0058


#### Quick Validation Check

In [9]:
# predictions on validation set
arr_valid_proba = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
arr_valid_pred = le.classes_[arr_valid_proba.argmax(axis=1)]

flt_valid_acc = accuracy_score(y_valid, arr_valid_pred)
flt_valid_logloss = log_loss(y_valid_enc, arr_valid_proba)

print(f'Validation accuracy: {flt_valid_acc:.4f}')
print(f'Validation log-loss: {flt_valid_logloss:.4f}')

Validation accuracy: 0.2339
Validation log-loss: 2.0058


#### Save Model and Artifacts Locally

In [10]:
# save model locally
str_filename = 'xgb_model.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
model.save_model(str_local_path)
print(f'Saved model to {str_local_path}')

# save label encoder locally
str_filename = 'label_encoder.joblib'
str_local_path = f'{str_dirname_output}/{str_filename}'
joblib.dump(le, str_local_path)
print(f'Saved label encoder to {str_local_path}')

# save best iteration for downstream use
str_filename = 'best_iteration.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump({'best_iteration': model.best_iteration}, f)
print(f'Saved best_iteration ({model.best_iteration}) to {str_local_path}')

# save test predictions locally
arr_test_proba = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
arr_test_pred = le.classes_[arr_test_proba.argmax(axis=1)]

df_test_preds = pd.DataFrame(arr_test_proba, columns=[f'prob_{c}' for c in le.classes_])
df_test_preds['predicted'] = arr_test_pred
df_test_preds['actual'] = y_test.values
str_filename = 'test_predictions.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_test_preds.to_csv(str_local_path, index=False)
print(f'Saved test predictions to {str_local_path}')

# save training params locally
dict_params_save = dict_params.copy()
dict_params_save['best_iteration'] = model.best_iteration
dict_params_save['best_score'] = model.best_score
str_filename = 'params.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump(dict_params_save, f, indent=2)
print(f'Saved params to {str_local_path}')

# save grid search results
str_filename = 'grid_search_results.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_results.to_csv(str_local_path, index=False)
print(f'Saved grid search results to {str_local_path}')

Saved model to ./output/xgb_model.json
Saved label encoder to ./output/label_encoder.joblib
Saved best_iteration (10) to ./output/best_iteration.json
Saved test predictions to ./output/test_predictions.csv
Saved params to ./output/params.json
Saved grid search results to ./output/grid_search_results.csv
